# Assignment 04 — Wine Data (Dataset: wine_data.csv)

EDA, regression, and classification for wine quality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report, accuracy_score
import joblib
import os

print('imports ok')

def ensure_models_dir():
    os.makedirs('models', exist_ok=True)


In [ ]:
# Load dataset

df = pd.read_csv('wine_data.csv')
print('shape:', df.shape)
df.head()

if 'quality' not in df.columns:
    raise ValueError('quality column not found')

# EDA
print('\nQuality distribution:')
print(df['quality'].value_counts())

# Regression: predict quality
X = df.drop(columns=['quality'])
y = df['quality']

# Simple impute numeric columns
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)

y_pred_reg = rf_reg.predict(X_test)
print('\nRegression RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_reg)))
print('R2:', r2_score(y_test, y_pred_reg))

# Classification (good vs bad)
threshold = df['quality'].median()
y_bin = (df['quality'] >= threshold).astype(int)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_bin, test_size=0.2, random_state=42, stratify=y_bin)

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train_c, y_train_c)

y_pred_c = rf_clf.predict(X_test_c)
print('\nClassification accuracy:', accuracy_score(y_test_c, y_pred_c))
print('\nClassification report:')
print(classification_report(y_test_c, y_pred_c, zero_division=0))

ensure_models_dir()
joblib.dump(rf_reg, 'models/wine_regressor.joblib')
joblib.dump(rf_clf, 'models/wine_classifier.joblib')
print('\nSaved regressor to models/wine_regressor.joblib and classifier to models/wine_classifier.joblib')
